# Demo Phi-3 Mini trên Colab T4

Notebook ngắn gọn theo `KH.pdf`: host `microsoft/Phi-3-mini-4k-instruct` trên GPU T4, hỏi một câu đơn giản, sau đó chạy benchmark Spark QA. Model tải khoảng 7.64 GB; nếu Colab runtime chưa reset thì lần sau sẽ đọc lại từ cache.

In [ ]:
# Chạy cell này trước khi import transformers/torch.
# Trên Colab, không cần restart runtime nếu các thư viện đã sẵn sàng.
%pip install -q "transformers>=4.40" "accelerate>=0.29" "huggingface_hub>=0.23"

## Lấy dữ liệu và mã benchmark từ GitHub

Cell này clone hoặc cập nhật repo vào `/content/a-triple-of-lms`. Benchmark sẽ lấy dataset từ `data/` và code từ `benchmarks/` trong repo này. Nếu chạy lại cell nhiều lần, notebook sẽ `git pull` thay vì clone lại.

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
PROJECT_DIR = Path("/content/a-triple-of-lms")

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

DATA_PATH = PROJECT_DIR / "data" / "spark_interview_questions.json"
BENCHMARK_DIR = PROJECT_DIR / "benchmarks"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dataset: {DATA_PATH}")
if not (BENCHMARK_DIR / "spark_qa_benchmark.py").exists():
    raise FileNotFoundError(f"Không tìm thấy benchmark script: {BENCHMARK_DIR / 'spark_qa_benchmark.py'}")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Benchmark:", BENCHMARK_DIR / "spark_qa_benchmark.py")

## Tùy chọn: lưu cache vào Google Drive

Colab sẽ mất cache khi runtime bị reset. Bật cell dưới đây nếu muốn lưu model vào Drive để các lần sau không phải tải lại 7.64 GB. Lần đầu vẫn phải tải một lần; các lần sau sẽ load từ `/content/drive/MyDrive/hf_cache`.

In [ ]:
import os

USE_GOOGLE_DRIVE_CACHE = True

if USE_GOOGLE_DRIVE_CACHE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        HF_CACHE_DIR = "/content/drive/MyDrive/hf_cache"
        os.makedirs(HF_CACHE_DIR, exist_ok=True)

        os.environ["HF_HOME"] = HF_CACHE_DIR
        os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
        os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "transformers")

        print("Hugging Face cache:", HF_CACHE_DIR)
    except ModuleNotFoundError:
        print("Không phải Colab, dùng cache mặc định của máy local.")

In [ ]:
import gc
import os
import time

# Chỉ cần cho Windows/Anaconda; trên Colab dòng này không ảnh hưởng.
if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU. Phi-3 mini vẫn có thể chạy CPU nhưng rất chậm.")

In [ ]:
# Load tokenizer + model. Cell này có thể chạy lại sau lỗi mà không cần restart runtime.
# Quan trọng: không dùng trust_remote_code cho Phi-3 trên Colab.
# Remote code cũ của Phi-3 dễ gây lỗi RoPE: KeyError 'type' hoặc Unknown RoPE scaling type.

if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

load_kwargs = {}

if torch.cuda.is_available():
    load_kwargs.update({
        "device_map": "cuda",
        "torch_dtype": torch.float16,
    })
else:
    load_kwargs.update({
        "device_map": "cpu",
        "torch_dtype": torch.float32,
    })

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

In [ ]:
def ask_phi3(question: str, max_new_tokens: int = 120) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    print(f"Latency: {latency:.1f}s")
    return answer

In [ ]:
question = "Explain MapReduce in exactly 5 bullet points. Each bullet must contain no more than 15 words."

print("Question:")
print(question)
print("\nPhi-3 output:")
print(ask_phi3(question))

## Benchmark Spark QA dataset

Phần này dùng `data/spark_interview_questions.json` và logic trong `benchmarks/spark_qa_benchmark.py` để chấm Phi-3. Nên chạy `BENCHMARK_LIMIT = 5` trước; full dataset hiện có 80 câu sẽ tốn thời gian trên T4.

In [ ]:
# Kiểm tra lại repo đã được clone và đưa project vào Python path.
from pathlib import Path
import sys

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("/content/a-triple-of-lms")

DATA_PATH = PROJECT_DIR / "data" / "spark_interview_questions.json"
BENCHMARK_DIR = PROJECT_DIR / "benchmarks"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy dataset. Hãy chạy cell clone GitHub ở phía trên trước. "
        f"Đường dẫn đang kiểm tra: {DATA_PATH}"
    )
if not (BENCHMARK_DIR / "spark_qa_benchmark.py").exists():
    raise FileNotFoundError(
        "Không tìm thấy benchmark script. Hãy chạy cell clone GitHub ở phía trên trước. "
        f"Đường dẫn đang kiểm tra: {BENCHMARK_DIR / 'spark_qa_benchmark.py'}"
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Benchmark:", BENCHMARK_DIR / "spark_qa_benchmark.py")

In [ ]:
import json
from pathlib import Path

from benchmarks.spark_qa_benchmark import (
    build_prompt,
    composite_score,
    generate_answer,
    save_visualizations,
    summarize,
    write_csv,
)

BENCHMARK_LIMIT = 5  # Đổi thành 0 để chạy full 80 câu.
MAX_NEW_TOKENS = 220
OUTPUT_DIR = PROJECT_DIR / "benchmark_results"

dataset = json.loads(DATA_PATH.read_text(encoding="utf-8"))
if BENCHMARK_LIMIT:
    dataset = dataset[:BENCHMARK_LIMIT]

print("Benchmark questions:", len(dataset))

In [ ]:
results = []

for index, item in enumerate(dataset, 1):
    prompt = build_prompt(item["question"])
    model_answer, latency_s, output_tokens = generate_answer(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    scores = composite_score(model_answer, item["reference_answer"])
    result = {
        "id": index,
        "question": item["question"],
        "category": item["category"],
        "difficulty": item["difficulty"],
        "reference_answer": item["reference_answer"],
        "model_answer": model_answer,
        "latency_s": round(latency_s, 3),
        "output_tokens": output_tokens,
        **scores,
    }
    results.append(result)
    print(
        f"[{index:03d}/{len(dataset):03d}] "
        f"{item['difficulty']} {item['category']} "
        f"score={result['score']:.4f} latency={result['latency_s']:.1f}s"
    )

summary = summarize(results)
summary["overall"]

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

details_path = OUTPUT_DIR / "spark_qa_benchmark_results.json"
summary_path = OUTPUT_DIR / "spark_qa_benchmark_summary.json"
csv_path = OUTPUT_DIR / "spark_qa_benchmark_results.csv"

details_path.write_text(json.dumps(results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
write_csv(results, csv_path)
chart_paths = save_visualizations(results, summary, OUTPUT_DIR)

print("Wrote:", details_path)
print("Wrote:", summary_path)
print("Wrote:", csv_path)

In [ ]:
# Hien thi charts trong notebook.
from IPython.display import Image, display

for chart_path in chart_paths:
    print(chart_path)
    display(Image(filename=str(chart_path)))

In [ ]:
# Xem nhanh câu điểm cao/thấp nhất để đưa vào slide demo.
ranked = sorted(results, key=lambda row: row["score"])

print("Lowest score:")
print(json.dumps(ranked[0], ensure_ascii=False, indent=2)[:2000])

print("\nHighest score:")
print(json.dumps(ranked[-1], ensure_ascii=False, indent=2)[:2000])